In [0]:
# Create the Volume first before writing data
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.customer360_bronze.raw_files")

# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("custId", IntegerType(), False),
    StructField("custName", StringType(), False),
    StructField("status", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
])

data = [ (1001,'John Smith','Active','Dallas','TX'),(1002,'Mary Jones','Active','Plano','TX'),(1003,'Steve Brown','Active','Frisco','TX')]
df = spark.createDataFrame(data, schema)

# Write data to Volume location as Delta format
volume_path = "/Volumes/workspace/customer360_bronze/raw_files/customers_01"
df.write.format("parquet").mode("overwrite").save(volume_path)

data2 = [ (1004,'Sarah Wilson','Active','Irving','TX'),(1005,'David Lee','Active','McKinney','TX')]
df = spark.createDataFrame(data2, schema)
volume_path2 = "/Volumes/workspace/customer360_bronze/raw_files/customers_02"
df.write.format("parquet").mode("overwrite").save(volume_path2)

print(f"Data written to {volume_path}")
print(f"Data written to {volume_path2}")


In [0]:
%sql
SELECT * FROM read_files('/Volumes/workspace/customer360_bronze/raw_files');

In [0]:
# This creates a Delta dataset containing Parquet files and a _delta_log transaction log.- created Delta datasets, not Parquet source directories
# That means Auto Loader is discovering Parquet files, not reading the Delta transaction log as a Delta table.
#It may successfully ingest the underlying Parquet files, but this is not a recommended production design. If you later overwrite or update those Delta datasets, Auto Loader can process new physical files without respecting Delta's transaction semantics. 

# USE Cell 4 CODE##############################################################################################################

%python

source_path = "/Volumes/workspace/customer360_bronze/raw_files"

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/"
    "raw_files/_checkpoints/customers"
)

schema_path = (
    "/Volumes/workspace/customer360_bronze/"
    "raw_files/_schemas/customers"
)

dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.customer360_bronze.customers_incremental")
)

query.awaitTermination()

In [0]:
%sql
select * from  workspace.customer360_bronze.customers_incremental/Volumes/workspace/customer360_bronze/raw_files;


you would need 2 path -->
1. store raw files/ parque file
2. checkpoint path. this must be different path from raw files

Although Auto Loader generally ignores hidden directories beginning with _, it's better to separate ingestion data from streaming metadata.

`````
/Volumes/workspace/customer360_bronze/
│
└── raw_files/
    ├── customers_01/
    │   └── part-....parquet
    ├── customers_02/
    │   └── part-....parquet
    ├── checkpoints/
    │   └── customers/
    └── schemas/
        └── customers/
        ````

In [0]:
from pyspark.sql import functions as F

# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("custId", IntegerType(), False),
    StructField("custName", StringType(), False),
    StructField("status", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
])

# Separate volume for streaming metadata
spark.sql("""
CREATE VOLUME IF NOT EXISTS
workspace.customer360_bronze.ingestion_metadata
""")

spark.sql("""
CREATE VOLUME IF NOT EXISTS
workspace.customer360_bronze.raw_files
""")

# New source directory for the Parquet exercise
source_path = (
    "/Volumes/workspace/customer360_bronze/raw_files/parquet_source"
)

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customers"
)

schema_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/schemas/customers"
)

# Create the initial source files
df1 = spark.createDataFrame(
    [
        (1001, "John Smith", "Active", "Dallas", "TX"),
        (1002, "Mary Jones", "Active", "Plano", "TX"),
        (1003, "Steve Brown", "Active", "Frisco", "TX")
    ],
    schema
)

df2 = spark.createDataFrame(
    [
        (1004, "Sarah Wilson", "Active", "Irving", "TX"),
        (1005, "David Lee", "Active", "McKinney", "TX")
    ],
    schema
)

df1.write.mode("overwrite").parquet(
    f"{source_path}/customers_01"
)

df2.write.mode("overwrite").parquet(
    f"{source_path}/customers_02"
)

# Configure Auto Loader
dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

# Ingest into Bronze
query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_bronze.customers_incremental_day09"
    )
)

query.awaitTermination()

# Verify
display(
    spark.table(
        "workspace.customer360_bronze.customers_incremental_day09"
    ).orderBy(F.col("custId"))
)

In [0]:
#Adding 3rd Parquet file to the source directory
df3 = spark.createDataFrame(
    [
        (1006, "Robert King", "Active", "Dallas", "TX"),
        (1007, "Jennifer White", "Active", "Plano", "TX")
    ],
    schema
)

df3.write.mode("overwrite").parquet(
    f"{source_path}/customers_03"
)

# Configure Auto Loader
dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

# Ingest into Bronze
query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_bronze.customers_incremental_day09"
    )
)

query.awaitTermination()

# Verify
display(
    spark.table(
        "workspace.customer360_bronze.customers_incremental_day09"
    ).orderBy(F.col("custId"))
)

In [0]:
from pyspark.sql import functions as F

# Use the SAME source and metadata paths
source_path = ("/Volumes/workspace/customer360_bronze/raw_files/parquet_source")
target_table = ("workspace.customer360_bronze.customers_incremental_day09")

checkpoint_path = ( "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customers")
schema_path = ("/Volumes/workspace/customer360_bronze/ingestion_metadata/schemas/customers")


# Auto Loader with schema evolution
dfAutoLoader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

# Write to Delta with schema merging enabled
query = (
    dfAutoLoader.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

# Validate
display(
    spark.table(target_table)
    .orderBy("custId")
)

In [0]:
%sql
SELECT *
FROM workspace.customer360_bronze.customers_incremental_day09
ORDER BY custId;

-- drop table workspace.customer360_bronze.customers_incremental_day09;
-- drop volume workspace.customer360_bronze.ingestion_metadata



In [0]:
%sql
SELECT * FROM read_files('/Volumes/workspace/customer360_bronze/raw_files/parquet_source/customers_01');

SELECT * FROM read_files('/Volumes/workspace/customer360_bronze/raw_files/parquet_source/customers_02');

SELECT * FROM read_files('/Volumes/workspace/customer360_bronze/raw_files/parquet_source/customers_03');

### Adding new field to the parquet file- email

when you make the change to teh defination (adding column- email) it will throw an error or stops the current schema with _UNKNOWN_FIELD_EXCEPTION. _ 
TO PASS THROUGH make 2 changes in the auto loaded code-
    1. **readstream** changes--> add .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    2. **writeStream** --> .option("mergeSchema", "true")



### Why did we add mergeSchema?
There are two separate schema evolution steps:
**Component	                Setting	                                            Purpose**
Auto Loader source	        cloudFiles.schemaEvolutionMode = addNewColumns	    Detects new columns in incoming files
Delta target	            mergeSchema = true	                                Allows the target table schema to include new columns

Both matter. Auto Loader can recognize the new column, but the Delta target must also accept it.

In [0]:
# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

source_path = ("/Volumes/workspace/customer360_bronze/raw_files/parquet_source")
target_table = ("workspace.customer360_bronze.customers_incremental_day09")

# add new column- emailId
schema_email = StructType([
    StructField("custId", IntegerType(), False),
    StructField("custName", StringType(), False),
    StructField("status", StringType(), False),
    StructField("city", StringType(), False),
    StructField("state", StringType(), False),
    StructField("emailId", StringType(), False)
])

df4 = spark.createDataFrame(
    [
        (1008, "Michael Scott", "Active", "Dallas", "TX", "michael@example.com"),
        (1009, "Pam Beesly", "Active", "Plano", "TX", "pam@example.com")
    ], schema_email
)

df4.write.mode("overwrite").parquet(f"{source_path}/customers_04")